In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS automobile_repair.silver;
CREATE OR REPLACE TABLE automobile_repair.silver.customer_survey AS
SELECT
    survey_id,
    order_id,

    -- Dates
    TRY_TO_DATE(survey_sent_date, 'yyyy-MM-dd') AS survey_sent_date,
    TRY_TO_DATE(survey_response_date, 'yyyy-MM-dd') AS survey_response_date,

    -- Response flag
    CASE 
        WHEN survey_response_date IS NOT NULL THEN 1
        ELSE 0
    END AS responded_flag,

    -- Ratings
    CAST(delivered_on_time_rating AS INT) AS delivered_on_time_rating,
    CAST(work_quality_rating AS INT) AS work_quality_rating,
    CAST(cleanliness_rating AS INT) AS cleanliness_rating,
    CAST(communication_rating AS INT) AS communication_rating,
    CAST(overall_satisfaction_rating AS INT) AS overall_satisfaction_rating,

    -- NEW COLUMN 
    ROUND(
        (
            COALESCE(delivered_on_time_rating,0) +
            COALESCE(work_quality_rating,0) +
            COALESCE(cleanliness_rating,0) +
            COALESCE(communication_rating,0) +
            COALESCE(overall_satisfaction_rating,0)
        ) / 5.0, 2
    ) AS average_rating

FROM (
    SELECT *,
           ROW_NUMBER() OVER (PARTITION BY survey_id ORDER BY survey_id) as rn
    FROM automobile_repair.bronze.customer_survey
)
WHERE rn = 1;